# Phase 1 — EEG→Image zero-shot retrieval baseline (THINGS-EEG2)
Run top to bottom on a Colab T4. Sections 1–7 use one subject; section 8 runs all 10 subjects.
Results reported in `README.md` come from `results/*.csv`.

## 1. Config

In [ ]:
# ===================== 0. CONFIG =====================
import os, sys, gc, json, math, time, random, importlib.util, subprocess
ROOT  = os.environ.get("EEG_ROOT",  "/content/EEG_Data")   # data folder
CACHE = os.environ.get("EEG_CACHE", "/content/cache")      # cached features/results
SUBJECT       = "sub-01"     # source subject (trained on)
CROSS_SUBJECT = "sub-02"     # only its TEST file is downloaded (for cross-subject check)

# Image/text encoder is a *candidate*, not a fixed choice (ablation-ready):
#   ("ViT-B-32","openai") fast | ("ViT-L-14","openai") | ("ViT-H-14","laion2b_s32b_b79k") strongest, slower
CLIP_ARCH, CLIP_PRETRAINED = "ViT-B-32", "openai"

EPOCHS, BATCH, LR, WD, PATIENCE = 40, 256, 3e-4, 1e-4, 10
SEEDS = [0, 1, 2]                 # use [0] if short on time
TEXT_WEIGHTS = [0.0, 0.1]         # 0.0 = image-only ; 0.1 = small text regularizer (MindAlign found 0.1 best, 0.5 hurts)
os.makedirs(CACHE, exist_ok=True)
# Optional: copy only the small result files (csv/png, a few MB) to Google Drive — any account works.
# Mount first:  from google.colab import drive; drive.mount('/content/drive')
RESULTS_DIR = None   # e.g. "/content/drive/MyDrive/eeg_results"
def backup():
    if RESULTS_DIR:
        import shutil; os.makedirs(RESULTS_DIR, exist_ok=True)
        for f in os.listdir(CACHE):
            if f.endswith((".csv", ".png")): shutil.copy(os.path.join(CACHE, f), RESULTS_DIR)

import numpy as np
for pkg, pip_name in [("open_clip", "open_clip_torch"), ("huggingface_hub", "huggingface_hub"), ("pandas", "pandas")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True   # same seed -> same numbers
    torch.backends.cudnn.benchmark = False

## 2. Download data (sub-01 + sub-02 test)

In [ ]:
from pathlib import Path
EEG_DIR = f"{ROOT}/Preprocessed_data_250Hz"
need = [f"{EEG_DIR}/{SUBJECT}/preprocessed_eeg_training.npy",
        f"{EEG_DIR}/{SUBJECT}/preprocessed_eeg_test.npy",
        f"{EEG_DIR}/{CROSS_SUBJECT}/preprocessed_eeg_test.npy"]
if not all(os.path.exists(p) for p in need):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="LidongYang/EEG_Image_decode", repo_type="dataset", local_dir=ROOT,
                      allow_patterns=[f"Preprocessed_data_250Hz/{SUBJECT}/*",
                                      f"Preprocessed_data_250Hz/{CROSS_SUBJECT}/preprocessed_eeg_test.npy"])
if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
    import tarfile
    from huggingface_hub import hf_hub_download
    tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz",
                               repo_type="dataset", local_dir=ROOT)
    with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
for p in need: print(("OK  " if os.path.exists(p) else "MISSING ") + p)

## 3. Average repetitions, cache as float32 (memory-mapped)

In [ ]:
def load_obj(path):
    # works whether the file was saved with np.save (0-d object array) or with pickle (plain dict)
    obj = np.load(path, allow_pickle=True)
    return obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj

def prep_eeg(subject, split, chunk=500):
    out_path = f"{CACHE}/{subject}_{split}_avg_f32.npy"
    if os.path.exists(out_path):
        return np.load(out_path, mmap_mode="r")
    src = f"{EEG_DIR}/{subject}/preprocessed_eeg_{'training' if split == 'train' else 'test'}.npy"
    t0 = time.time()
    d = load_obj(src)
    arr = d["preprocessed_eeg_data"]                       # (N, reps, C, T) float64
    out = np.empty((arr.shape[0],) + arr.shape[2:], dtype=np.float32)
    for i in range(0, arr.shape[0], chunk):
        out[i:i + chunk] = arr[i:i + chunk].mean(axis=1)    # average repetitions chunk by chunk
    reps = arr.shape[1]
    del d, arr; gc.collect()
    np.save(out_path, out)
    print(f"{subject}/{split}: {out.shape} (averaged over {reps} reps) in {time.time()-t0:.0f}s")
    return out

Xtr_raw = prep_eeg(SUBJECT, "train")
Xte_raw = prep_eeg(SUBJECT, "test")
Xx_raw  = prep_eeg(CROSS_SUBJECT, "test")
N_TR, N_CH, N_T = Xtr_raw.shape
print("train", Xtr_raw.shape, "| test", Xte_raw.shape, "| cross-subject test", Xx_raw.shape)

## 4. Frozen CLIP image/text features (replaceable target)

In [ ]:
import open_clip
from PIL import Image
from torch.utils.data import Dataset, DataLoader

meta = load_obj(f"{ROOT}/images_set/image_metadata.npy")
train_files, test_files = list(meta["train_img_files"]), list(meta["test_img_files"])
assert len(train_files) == N_TR and len(test_files) == len(Xte_raw), "EEG/image count mismatch"

def build_index(folder):
    idx = {}
    for dp, _, fs in os.walk(folder):
        for f in fs: idx.setdefault(f, os.path.join(dp, f))
    return idx
img_root = f"{ROOT}/images_set"
idx_train = build_index(f"{img_root}/training_images")
idx_test  = build_index(f"{img_root}/test_images") if os.path.isdir(f"{img_root}/test_images") else build_index(img_root)
train_paths = [idx_train[f] for f in train_files]
test_paths  = [idx_test[f] for f in test_files]

def concept_names(key, n_imgs):
    c = list(meta[key])
    if len(c) != n_imgs:                          # one entry per concept -> expand to images
        per = n_imgs // len(c); c = [x for x in c for _ in range(per)]
    clean = lambda s: s.split("_", 1)[1].replace("_", " ") if s.split("_", 1)[0].isdigit() else s.replace("_", " ")
    return [clean(x) for x in c]
train_concepts = concept_names("train_img_concepts", N_TR)
test_concepts  = concept_names("test_img_concepts", len(test_files))
uniq = sorted(set(train_concepts)); c2i = {c: i for i, c in enumerate(uniq)}
train_labels = np.array([c2i[c] for c in train_concepts])
print(f"{len(uniq)} train concepts, {len(set(test_concepts))} test concepts, overlap = {len(set(uniq) & set(test_concepts))} (should be 0 = zero-shot)")

tag = f"{CLIP_ARCH}_{CLIP_PRETRAINED}".replace("/", "-")
feat_path = f"{CACHE}/feats_{tag}.pt"
if os.path.exists(feat_path):
    F_ = torch.load(feat_path)
else:
    clip_model, _, preprocess = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_PRETRAINED)
    clip_model = clip_model.to(device).eval()
    tokenizer = open_clip.get_tokenizer(CLIP_ARCH)
    class ImgDS(Dataset):
        def __init__(self, paths): self.paths = paths
        def __len__(self): return len(self.paths)
        def __getitem__(self, i): return preprocess(Image.open(self.paths[i]).convert("RGB"))
    @torch.no_grad()
    def encode_images(paths):
        out = []
        for x in DataLoader(ImgDS(paths), batch_size=128, num_workers=2):
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                out.append(F.normalize(clip_model.encode_image(x.to(device)).float(), dim=-1).cpu())
        return torch.cat(out)
    @torch.no_grad()
    def encode_texts(names):
        toks = tokenizer([f"a photo of a {n}" for n in names]).to(device)
        with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
            return F.normalize(clip_model.encode_text(toks).float(), dim=-1).cpu()
    t0 = time.time()
    F_ = {"train_img": encode_images(train_paths), "test_img": encode_images(test_paths),
          "train_txt": encode_texts(uniq), "test_txt": encode_texts(test_concepts)}
    torch.save(F_, feat_path); print(f"features done in {time.time()-t0:.0f}s")
    del clip_model; torch.cuda.empty_cache() if device == "cuda" else None
D_EMB = F_["train_img"].shape[1]
print({k: tuple(v.shape) for k, v in F_.items()})

## 5. Within-subject training: image-only vs + text regularizer (3 seeds)

In [ ]:
# channel-wise z-score with TRAIN statistics (applied to every split / subject-specific for cross-subject)
mu = Xtr_raw.mean(axis=(0, 2), keepdims=True).astype(np.float32)
sd = Xtr_raw.std(axis=(0, 2), keepdims=True).astype(np.float32) + 1e-6
to_t = lambda a, m, s: torch.from_numpy(((np.asarray(a) - m) / s).astype(np.float32))
Xtr = to_t(Xtr_raw, mu, sd).to(device)
Xte = to_t(Xte_raw, mu, sd).to(device)
# cross-subject: normalize with that subject's own (unlabeled) channel statistics
Xx  = to_t(Xx_raw, Xx_raw.mean(axis=(0, 2), keepdims=True), Xx_raw.std(axis=(0, 2), keepdims=True) + 1e-6).to(device)

per = N_TR // len(uniq)
is_val = (np.arange(N_TR) % per) == (per - 1)
tr_idx, va_idx = np.where(~is_val)[0], np.where(is_val)[0]
IMG_tr, IMG_te = F_["train_img"].to(device), F_["test_img"].to(device)
TXT_tr = F_["train_txt"].to(device)
LBL = torch.from_numpy(train_labels).to(device)
print(f"train {len(tr_idx)} | val {len(va_idx)} | test {len(Xte)}")

class EEGEncoder(nn.Module):
    def __init__(self, n_ch, n_t, out_dim, k=40, drop=0.5):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, k, (1, 25)), nn.AvgPool2d((1, 51), (1, 5)), nn.BatchNorm2d(k), nn.ELU(),
            nn.Conv2d(k, k, (n_ch, 1)), nn.BatchNorm2d(k), nn.ELU(), nn.Dropout(drop))
        with torch.no_grad(): flat = self.backbone(torch.zeros(1, 1, n_ch, n_t)).numel()
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(flat, out_dim))
        self.res = nn.Sequential(nn.GELU(), nn.Linear(out_dim, out_dim), nn.Dropout(drop))
        self.norm = nn.LayerNorm(out_dim)
    def forward(self, x):
        h = self.proj(self.backbone(x.unsqueeze(1)))
        return F.normalize(self.norm(h + self.res(h)), dim=-1)

@torch.no_grad()
def embed(model, X, bs=512):
    model.eval(); return torch.cat([model(X[i:i + bs]) for i in range(0, len(X), bs)])

@torch.no_grad()
def retrieval(z, targets, ks=(1, 5)):
    # z[i] must retrieve targets[i] among all rows of targets (N-way)
    sim = z @ targets.T
    rank = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {f"top{k}": (rank < k).float().mean().item() * 100 for k in ks}

def train_run(text_weight, seed, verbose=True):
    set_seed(seed)
    model = EEGEncoder(N_CH, N_T, D_EMB).to(device)
    logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07), device=device))
    opt = torch.optim.AdamW(list(model.parameters()) + [logit_scale], lr=LR, weight_decay=WD)
    best, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train(); perm = torch.from_numpy(np.random.permutation(tr_idx)).to(device); tot = 0
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            z = model(Xtr[b]); s = logit_scale.exp().clamp(max=100)
            logits = s * z @ IMG_tr[b].T
            y = torch.arange(len(b), device=device)
            loss = (F.cross_entropy(logits, y) + F.cross_entropy(logits.T, y)) / 2
            if text_weight > 0:
                loss = loss + text_weight * F.cross_entropy(s * z @ TXT_tr.T, LBL[b])
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(b)
        va = retrieval(embed(model, Xtr[va_idx]), IMG_tr[va_idx])
        if va["top5"] > best:
            best, bad = va["top5"], 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}; best_ep = ep + 1
        else:
            bad += 1
        if verbose and (ep % 5 == 0 or ep == EPOCHS - 1):
            print(f"  ep {ep+1:2d} loss {tot/len(perm):.3f} | val({len(va_idx)}-way) top1 {va['top1']:.2f} top5 {va['top5']:.2f}")
        if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    te = retrieval(embed(model, Xte), IMG_te)
    return model, {"text_weight": text_weight, "seed": seed, "best_epoch": best_ep,
                   "val_top5": best, "test_top1": te["top1"], "test_top5": te["top5"]}

import pandas as pd
results, models = [], {}
for tw in TEXT_WEIGHTS:
    for sd_ in SEEDS:
        t0 = time.time(); print(f"== text_weight={tw} seed={sd_}")
        m, r = train_run(tw, sd_); r["minutes"] = round((time.time() - t0) / 60, 1)
        results.append(r); models[(tw, sd_)] = m
        print(f"   -> TEST 200-way: top1 {r['test_top1']:.2f}%  top5 {r['test_top5']:.2f}%  (best epoch {r['best_epoch']})")
res = pd.DataFrame(results); res.to_csv(f"{CACHE}/results_within_subject.csv", index=False)
summary = res.groupby("text_weight")[["test_top1", "test_top5"]].agg(["mean", "std"]).round(2)
print("\nChance: top1 0.50% | top5 2.50%"); print(summary)

## 6. SCORE sanity check: recover a known rotation (labelled vs label-free)

In [ ]:
def procrustes(X, Y, w=None, rho=0.0):
    # orthogonal R minimizing ||X R - Y||_F  (rows of X and Y are paired)
    w = torch.ones(len(X), 1, device=X.device) if w is None else w
    M = X.T @ (w * Y)
    if rho > 0: M = M + rho * torch.linalg.matrix_norm(M, ord=2) * torch.eye(M.shape[0], device=M.device)
    U, S, Vh = torch.linalg.svd(M)
    return U @ Vh                                    # <-- correct (old code had U @ Vh.T)

def rotation(d, angle_deg, seed=0):
    # rotate EVERY 2-D plane of a random orthonormal basis by exactly angle_deg
    # (cosine similarities shrink by cos(angle): 90 deg = alignment fully scrambled)
    g = torch.Generator().manual_seed(seed)
    U, _ = torch.linalg.qr(torch.randn(d, d, generator=g))
    t = math.radians(angle_deg); c, s = math.cos(t), math.sin(t)
    B = torch.eye(d)
    for i in range(0, d - 1, 2):
        B[i, i], B[i, i + 1], B[i + 1, i], B[i + 1, i + 1] = c, -s, s, c
    return (U @ B @ U.T).to(device)

def csls(X, Y, k=10):
    s = X @ Y.T
    return 2 * s - s.topk(k, dim=1).values.mean(1, keepdim=True) - s.topk(k, dim=0).values.mean(0, keepdim=True)

def unsupervised_align(Zt, Zs_ref, iters=5, k=10, rho=0.1):
    # iterative MNN(CSLS) landmarks + regularized Procrustes, starting from identity (SCORE-like refinement).
    # Safeguard: a new R is accepted only if it raises the mean CSLS matching score (never worse than identity).
    R = torch.eye(Zt.shape[1], device=Zt.device)
    fit = lambda R_: csls(Zt @ R_, Zs_ref, k).max(1).values.mean().item()
    best_fit, n_mut = fit(R), 0
    for _ in range(iters):
        c = csls(Zt @ R, Zs_ref, k)
        a, b = c.argmax(1), c.argmax(0)
        mutual = torch.where(b[a] == torch.arange(len(a), device=a.device))[0]
        if len(mutual) < 10: break
        R_new = procrustes(Zt[mutual], Zs_ref[a[mutual]], rho=rho)
        f = fit(R_new)
        if f <= best_fit: break
        R, best_fit, n_mut = R_new, f, len(mutual)
    return R, n_mut

base = models[(TEXT_WEIGHTS[0], SEEDS[0])]
Zs_val, Zs_te = embed(base, Xtr[va_idx]), embed(base, Xte)
ref = retrieval(Zs_te, IMG_te)["top1"]
Q, _ = torch.linalg.qr(torch.randn(D_EMB, D_EMB, generator=torch.Generator().manual_seed(0)))
Q = Q.to(device)                                     # Haar-random FULL rotation (like a brand-new subject)
print(f"(a) Reference (no rotation) test top1 = {ref:.2f}%  | after FULL random rotation = {retrieval(Zs_te @ Q, IMG_te)['top1']:.2f}%")
score_rows = []
for n_lm in sorted({min(n, len(va_idx)) for n in [10, 50, 100, 250, 500, 1000, len(va_idx)]}):
    lm = torch.randperm(len(va_idx))[:n_lm]
    R = procrustes((Zs_val @ Q)[lm], Zs_val[lm])
    acc = retrieval(F.normalize(Zs_te @ Q @ R, dim=-1), IMG_te)["top1"]
    score_rows.append({"exp": "labelled landmarks", "param": n_lm, "test_top1": acc})
    print(f"    {n_lm:5d} labelled landmarks -> test top1 {acc:.2f}%")
print("\n(b) Unsupervised (MNN+CSLS) recovery vs. rotation size")
print("    target = rotated EEG of val half A ; reference = un-rotated EEG of val half B (different images, no pairing)")
half = len(va_idx) // 2
for ang in [0, 10, 20, 30, 45, 60, 90]:
    Qa = rotation(D_EMB, ang, seed=1)
    before = retrieval(Zs_te @ Qa, IMG_te)["top1"]
    R, n_mut = unsupervised_align(Zs_val[:half] @ Qa, Zs_val[half:])
    after = retrieval(F.normalize(Zs_te @ Qa @ R, dim=-1), IMG_te)["top1"]
    score_rows.append({"exp": "unsupervised", "param": ang, "before": before, "test_top1": after, "mutual": n_mut})
    print(f"    angle {ang:3d}° | before {before:5.2f}% | after {after:5.2f}% | mutual landmarks {n_mut}")
pd.DataFrame(score_rows).to_csv(f"{CACHE}/results_score.csv", index=False)

## 7. Cross-subject sub-01 → sub-02 (100-way): zero-shot, moment matching, Procrustes

In [ ]:
E = torch.arange(len(Xte) // 2, len(Xte), device=device)     # evaluation half
C = torch.arange(0, len(Xte) // 2, device=device)            # calibration half
xs_rows = []
for (tw, s_), m in models.items():
    z_src, z_tgt = embed(m, Xte), embed(m, Xx)
    z_val = embed(m, Xtr[va_idx])
    row = {"text_weight": tw, "seed": s_,
           "within_sub01": retrieval(z_src[E], IMG_te[E])["top1"],
           "zero_shot_sub02": retrieval(z_tgt[E], IMG_te[E])["top1"]}
    z_mm = F.normalize(z_tgt - z_tgt.mean(0) + z_val.mean(0), dim=-1)     # unsupervised moment matching
    row["moment_match_sub02"] = retrieval(z_mm[E], IMG_te[E])["top1"]
    for k in [25, 50, 100]:
        R = procrustes(z_mm[C[:k]], z_src[C[:k]], rho=0.5)                 # map sub-02 -> sub-01 EEG space
        row[f"procrustes_k{k}_sub02"] = retrieval(F.normalize(z_mm[E] @ R, dim=-1), IMG_te[E])["top1"]
    xs_rows.append(row)
xs = pd.DataFrame(xs_rows); xs.to_csv(f"{CACHE}/results_cross_subject.csv", index=False)
print("100-way top1 (%), chance = 1%"); print(xs.groupby("text_weight").mean(numeric_only=True).drop(columns="seed").round(2).T)

## 7b. Figure + summary printout

In [ ]:
import matplotlib.pyplot as plt
g = res.groupby("text_weight")[["test_top1", "test_top5"]]
mean, std = g.mean(), g.std().fillna(0)
labels = ["EEG→Image" if tw == 0 else f"EEG→Image + Text (λ={tw})" for tw in mean.index]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(labels))
for j, (col, ch) in enumerate([("test_top1", 0.5), ("test_top5", 2.5)]):
    ax[j].bar(x, mean[col], yerr=std[col], color="#4C72B0", capsize=4, width=0.55)
    ax[j].axhline(ch, ls="--", color="gray"); ax[j].text(len(x) - 0.5, ch, " chance", va="bottom", color="gray", ha="right")
    ax[j].set_xticks(x, labels, fontsize=9); ax[j].set_ylabel("%"); ax[j].set_title(f"{col.replace('test_', '').upper()} — 200-way zero-shot, {SUBJECT}")
    ax[j].spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(f"{CACHE}/fig_within_subject.png", dpi=200); plt.show()

print("==== COPY EVERYTHING BELOW AND SEND IT BACK ====")
print(f"encoder={CLIP_ARCH}/{CLIP_PRETRAINED} seeds={SEEDS}")
print(res.round(2).to_string(index=False))
print(pd.read_csv(f"{CACHE}/results_score.csv").round(2).to_string(index=False))
print(xs.round(2).to_string(index=False))
backup()

## 8-A. Prepare all 10 subjects (one subprocess each, avoids RAM build-up)

In [ ]:
# 8-A: prepare all subjects, each in its own process (no RAM build-up). Run right after cells 1-2.
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
PREP_SCRIPT = '''
import numpy as np, os, sys, time
eeg_dir, cache, sub, split = sys.argv[1:5]
src = f"{eeg_dir}/{sub}/preprocessed_eeg_{'training' if split == 'train' else 'test'}.npy"
out_path = f"{cache}/{sub}_{split}_avg_f32.npy"
t0 = time.time()
obj = np.load(src, allow_pickle=True)
d = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
arr = d["preprocessed_eeg_data"]
out = np.empty((arr.shape[0],) + arr.shape[2:], dtype=np.float32)
for i in range(0, arr.shape[0], 500):
    out[i:i + 500] = arr[i:i + 500].mean(axis=1)
np.save(out_path + ".tmp.npy", out); os.replace(out_path + ".tmp.npy", out_path)
print(f"  {sub}/{split}: {out.shape} in {time.time()-t0:.0f}s", flush=True)
'''
for sub in ALL_SUBJECTS:
    for split in ["train", "test"]:
        out_path = f"{CACHE}/{sub}_{split}_avg_f32.npy"
        if os.path.exists(out_path): continue
        fname = f"preprocessed_eeg_{'training' if split == 'train' else 'test'}.npy"
        raw = f"{EEG_DIR}/{sub}/{fname}"
        if not os.path.exists(raw):
            from huggingface_hub import hf_hub_download
            hf_hub_download(repo_id="LidongYang/EEG_Image_decode", repo_type="dataset", local_dir=ROOT,
                            filename=f"Preprocessed_data_250Hz/{sub}/{fname}")
        r = subprocess.run([sys.executable, "-c", PREP_SCRIPT, EEG_DIR, CACHE, sub, split], capture_output=True, text=True)
        print(r.stdout.strip())
        if r.returncode != 0:
            print(r.stderr[-1500:]); raise RuntimeError(f"preparing {sub}/{split} failed (message above)")
        if os.environ.get("EEG_KEEP_RAW") != "1" and os.path.exists(raw): os.remove(raw)
    print(f"{sub} ready")
print("All subjects prepared:", len([f for f in os.listdir(CACHE) if f.endswith("_avg_f32.npy")]), "files")

## 8-B. Within-subject for every subject + 10×10 cross-subject matrix
Note: diagonal here normalizes each test set with its **own** statistics, so it differs from the within-subject numbers (train-statistics normalization).

In [ ]:
# 8-B: within-subject training for every subject + 10x10 cross-subject matrix (resumable).
import matplotlib.pyplot as plt
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
ALL_SEED, ALL_TEXT_WEIGHT = 0, 0.0
all_rows, sub_models, sub_test = [], {}, {}
for sub in ALL_SUBJECTS:
    t0 = time.time()
    tr_raw = np.load(f"{CACHE}/{sub}_train_avg_f32.npy", mmap_mode="r")
    te_raw = np.load(f"{CACHE}/{sub}_test_avg_f32.npy", mmap_mode="r")
    sub_test[sub] = to_t(te_raw, te_raw.mean(axis=(0, 2), keepdims=True), te_raw.std(axis=(0, 2), keepdims=True) + 1e-6)
    m_path, r_path = f"{CACHE}/model_{sub}.pt", f"{CACHE}/result_{sub}.json"
    if os.path.exists(m_path) and os.path.exists(r_path):
        model = EEGEncoder(N_CH, N_T, D_EMB); model.load_state_dict(torch.load(m_path, map_location="cpu"))
        r = json.load(open(r_path)); print(f"{sub}: loaded from cache")
    else:
        m_, s_ = tr_raw.mean(axis=(0, 2), keepdims=True), tr_raw.std(axis=(0, 2), keepdims=True) + 1e-6
        del Xtr; gc.collect(); torch.cuda.empty_cache() if device == "cuda" else None
        Xtr, Xte = to_t(tr_raw, m_, s_).to(device), to_t(te_raw, m_, s_).to(device)   # globals used by train_run
        model, r = train_run(ALL_TEXT_WEIGHT, ALL_SEED, verbose=False)
        r["subject"] = sub; r["minutes"] = round((time.time() - t0) / 60, 1)
        torch.save(model.state_dict(), m_path); json.dump(r, open(r_path, "w"))
    sub_models[sub] = model.cpu(); all_rows.append(r)
    print(f"{sub}: test 200-way top1 {r['test_top1']:.2f}%  top5 {r['test_top5']:.2f}%  ({r.get('minutes', 0)} min)")
    pd.DataFrame(all_rows).to_csv(f"{CACHE}/results_all_subjects.csv", index=False); backup()

subs = list(sub_models)
mat = np.zeros((len(subs), len(subs)))
for i, si in enumerate(subs):
    m = sub_models[si].to(device)
    for j, sj in enumerate(subs):
        mat[i, j] = retrieval(embed(m, sub_test[sj].to(device)), IMG_te)["top1"]
    sub_models[si] = m.cpu()
cross = pd.DataFrame(mat, index=[f"train {s}" for s in subs], columns=[f"test {s}" for s in subs]).round(1)
cross.to_csv(f"{CACHE}/results_cross_matrix.csv")
off, diag = mat[~np.eye(len(subs), dtype=bool)], np.diag(mat)
print(f"\nWithin-subject top1 = {diag.mean():.2f} ± {diag.std(ddof=1):.2f}%  |  cross-subject (off-diagonal) top1 = {off.mean():.2f} ± {off.std(ddof=1):.2f}%  | chance 0.5%")
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mat, cmap="viridis"); plt.colorbar(im, ax=ax, label="Top-1 (%)")
ax.set_xticks(range(len(subs)), [s[-2:] for s in subs]); ax.set_yticks(range(len(subs)), [s[-2:] for s in subs])
ax.set_xlabel("test subject"); ax.set_ylabel("train subject"); ax.set_title("200-way zero-shot Top-1: within (diagonal) vs cross-subject")
for i in range(len(subs)):
    for j in range(len(subs)): ax.text(j, i, f"{mat[i,j]:.0f}", ha="center", va="center", color="w", fontsize=7)
plt.tight_layout(); plt.savefig(f"{CACHE}/fig_cross_matrix.png", dpi=200); plt.show()
print("==== COPY ====\n" + pd.DataFrame(all_rows)[["subject", "test_top1", "test_top5", "best_epoch"]].round(2).to_string(index=False))
print(cross.to_string())
backup()